# MVP de Engenharia de Dados — Etapa 5a: Análise da qualidade dos dados---O descritivo pede uma análise de qualidade **para cada atributo** do conjunto de dados: *"Existem problemas no conjunto de dados? Caso haja, como esses problemas podem ser resolvidos para que não afetem as respostas das perguntas que quer solucionar?"*O critério adotado é o da Aula 2 de Governança de Dados: avaliar **coluna a coluna** quais valores são esperados — domínio, faixa, formato, obrigatoriedade — e confrontar com o que existe. A apostila é clara sobre a dependência entre metadados e qualidade:> "Não é possível avaliar a qualidade de um conjunto de dados sem ter conhecimento dos seus metadados."Por isso cada verificação executada aqui declara, em texto, **o que se esperava encontrar** antes de mostrar o que encontrou. Os domínios esperados estão no [catálogo de dados](../docs/04-catalogo-de-dados.md).Todo o SQL está em [`sql/30_qualidade.sql`](../sql/30_qualidade.sql), e os resultados são **persistidos** em duas tabelas do BigQuery — a evidência fica gravada na plataforma, não apenas na saída deste notebook.

In [ ]:
PROJETO_ID = "mvp-criminalidade-sorocaba"
REPO_DIR   = "/content/mvp-engenharia-dados"
REPO_URL   = "https://github.com/SEU_USUARIO/mvp-engenharia-dados.git"

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import os

cliente_bq = bigquery.Client(project=PROJETO_ID)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 90)

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull --quiet
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}


def executar_arquivo_sql(nome_arquivo):
    with open(f"{REPO_DIR}/sql/{nome_arquivo}", encoding="utf-8") as arquivo:
        sql = arquivo.read().replace("@projeto", PROJETO_ID)
    cliente_bq.query(sql).result()
    print(f"{nome_arquivo}: concluído")


def consultar(sql):
    return cliente_bq.query(sql.replace("@projeto", PROJETO_ID)).to_dataframe()

In [ ]:
executar_arquivo_sql("30_qualidade.sql")

## 1. Perfil de cada atributoPara cada um dos 30 atributos da tabela conformada: quantos valores estão preenchidos, quantos são distintos, o menor e o maior, e quantos valores distintos deixariam de ser distintos se a acentuação e a caixa fossem ignoradas — indicador direto de "a mesma coisa escrita de formas diferentes".

In [ ]:
perfil = consultar("SELECT * FROM `@projeto.qualidade.perfil_atributos` ORDER BY percentual_nulos DESC")
perfil

In [ ]:
# Os atributos com preenchimento incompleto, em ordem
incompletos = perfil[perfil.percentual_nulos > 0].sort_values("percentual_nulos")

figura, eixo = plt.subplots(figsize=(9, max(3, 0.32 * len(incompletos))))
eixo.barh(incompletos.atributo, incompletos.percentual_nulos, color="#c0392b")
eixo.set_xlabel("% de valores ausentes")
eixo.set_title("Completude por atributo — tabela conformada de Sorocaba")
for y, (_, linha) in enumerate(incompletos.iterrows()):
    eixo.text(linha.percentual_nulos + 0.4, y, f"{linha.percentual_nulos:.1f}%",
              va="center", fontsize=8)
eixo.set_xlim(0, max(incompletos.percentual_nulos) * 1.15)
plt.tight_layout()
plt.show()

### Leitura do perfil**O que está íntegro.** A maior parte dos atributos tem preenchimento completo: identificação do boletim, datas, natureza criminal, rubrica, delegacia de circunscrição, área da Polícia Militar e código do município não têm um único valor ausente. Os campos que sustentam as perguntas P1, P2, P6 e P7 estão, portanto, completos.**Onde estão os problemas.** A incompletude se concentra em três campos, todos ligados a *quando* e *onde exatamente* o fato ocorreu:| Atributo | Situação na origem | Efeito nas perguntas ||---|---|---|| `periodo_ocorrencia` | ausente em mais da metade dos registros | seria fatal para P4 se não fosse tratado || `hora_ocorrencia` | ausente em cerca de um quarto | limita P4 || `latitude` / `longitude` | ausentes ou zeradas em cerca de 38% | limita P8 || `bairro` | ausente em cerca de 1,7% | efeito pequeno em P6 || `tipo_local` | **não publicado** em 2022, 2023 e 2024 | seria fatal para P5 |**A coluna `distintos_por_grafia`** mede o problema tratado pela padronização do ETL: quantos valores distintos colapsam quando acentuação e caixa são ignoradas. Diferente de zero significa que a fonte escreve a mesma coisa de formas diferentes — e que, sem tratamento, uma única categoria seria contada como várias.

## 2. Verificações com resultado esperado declarado

In [ ]:
verificacoes = consultar("""
SELECT verificacao, esperado, resultado, situacao
FROM `@projeto.qualidade.verificacoes`
ORDER BY CASE situacao WHEN 'ATENÇÃO' THEN 1 WHEN 'TRATADO' THEN 2
                       WHEN 'INFORMATIVO' THEN 3 ELSE 4 END, verificacao
""")
verificacoes

### Os problemas encontrados, e o que foi feito com cada umA seguir, cada problema com a evidência que o revelou, a decisão tomada e onde ela foi implementada. Os números são os da carga de agosto de 2026.---#### Problema 1 — A mesma natureza criminal escrita de formas diferentes**Evidência.** A origem traz **28 valores distintos** de natureza apurada, que representam apenas **23 naturezas reais**. `TRÁFICO DE ENTORPECENTES` aparece 1.207 vezes e `TRAFICO DE ENTORPECENTES`, sem acento, outras 376. `LESÃO CORPORAL CULPOSA - OUTRAS` usa hífen em 147 registros e travessão em 36.**Por que importa.** Sem tratamento, o tráfico de entorpecentes apareceria em dois lugares de qualquer ranking, com 1.207 e 376 ocorrências, e nenhum dos dois seria o número certo. É o tipo de erro que não gera exceção nem alerta — apenas uma resposta errada.**Decisão.** Padronizar para maiúsculas sem acento, com travessão convertido em hífen e espaços colapsados. O valor original é preservado em `natureza_apurada_origem`.**Onde:** [`spark/etl_ocorrencias.py`](../spark/etl_ocorrencias.py), transformação T4.---#### Problema 2 — O período do dia ausente em mais da metade dos registros**Evidência.** `DESC_PERIODO` está vazio em cerca de 54% dos registros. Mas a hora, em boa parte desses casos, está preenchida.**Por que importa.** A pergunta P4 depende inteiramente desse campo. Com metade dos dados fora, qualquer distribuição por período seria uma amostra não aleatória — e provavelmente enviesada, já que a ausência do período não é aleatória.**Decisão.** Derivar o período a partir da hora, usando as mesmas quatro faixas da fonte. A coluna `origem_periodo` distingue o publicado do derivado. Após a derivação, apenas **1.344 de 73.394 registros (1,8%)** ficam sem período — contra 54% na origem.**Ressalva mantida.** A categoria `EM HORA INCERTA`, que a fonte usa explicitamente, é preservada: ela é informação, não ausência.---#### Problema 3 — O tipo de local não existe em três dos cinco anos**Evidência.** `DESCR_TIPOLOCAL` só aparece nos arquivos de 2025 e 2026. Nos anos anteriores há apenas o subtipo.**Por que importa.** A pergunta P5 valeria para dois dos cinco anos.**Decisão.** Derivar o tipo a partir do subtipo, usando a correspondência observada nos anos que publicam os dois campos — calculada sobre o estado inteiro, para cobrir subtipos raros.**Limite declarado.** A derivação não é exata. Um subtipo é marcado como ambíguo quando o tipo predominante responde por menos de 95% das suas ocorrências. **A pergunta P5 é respondida apenas sobre os anos publicados**; o dado derivado fica disponível, marcado, para quem aceitar a imprecisão em troca da série completa.---#### Problema 4 — O bairro é campo livre, com grafias inconsistentes**Evidência.** O mesmo bairro aparece como `JARDIM SAO CARLOS`, `JD SAO CARLOS`, `JD. SAO CARLOS` e `JD.SAO CARLOS`. Sorocaba tem algumas centenas de bairros oficiais; a origem produz mais de mil grafias distintas.**Decisão.** Padronizar caixa e acentuação e expandir as abreviações de logradouro mais comuns (`JD` → `JARDIM`, `VL` → `VILA`, `PQ` → `PARQUE`, entre outras). O tratamento reduz a cardinalidade em cerca de 12%.**Limite declarado.** O tratamento **não resolve o problema por completo**: continuam existindo variações que só um cadastro oficial de bairros de Sorocaba resolveria — e esse cadastro não está disponível como dado aberto estruturado. Por isso a pergunta P6 é respondida prioritariamente por **delegacia de circunscrição**, que é um campo controlado com 11 valores, e o bairro é usado como detalhamento, não como eixo principal.---#### Problema 5 — Coordenadas ausentes ou zeradas**Evidência.** Cerca de 38% dos registros não têm geolocalização utilizável: a fonte usa `0`, `-` e `NULL` para representar ausência.**Decisão.** A coordenada `0` **não é** zero grau — é ausência disfarçada de número, e vira nulo, com a marcação `tem_geolocalizacao`. Dos registros que têm coordenada, **99,2% caem dentro da moldura geográfica de Sorocaba**, o que confirma que as coordenadas presentes são confiáveis.**Consequência para P8.** A pergunta é respondível, sobre a parcela geolocalizada — com a ressalva explícita de que ela não é necessariamente representativa do total.---#### Problema 6 — Horários arredondados**Evidência.** Entre os registros com hora informada, **41% caem exatamente no minuto 00 e 17% no minuto 30**. Se os horários fossem precisos, cada minuto responderia por cerca de 1,7%.**Por que importa.** É o achado mais sutil desta análise. Ele não é um erro a corrigir: é uma **propriedade da fonte**. A maioria dos horários é uma *estimativa* da vítima, não uma medição. A consequência prática é que analisar por hora cheia é legítimo, mas qualquer análise de granularidade menor — por minuto, ou comparando 14h10 com 14h40 — estaria lendo ruído como sinal.**Decisão.** Nenhuma correção. A dimensão de tempo do dia tem granularidade de **hora cheia**, e a limitação está declarada na análise de P4.---#### Problema 7 — O ano do arquivo não é o ano do fato**Evidência.** O arquivo de cada ano é fechado pelo mês de entrada na estatística oficial. Cerca de **3% dos fatos** ocorreram em ano diferente do da estatística, e o fato mais antigo encontrado é de **1976**.**Por que importa.** Uma série anual construída sobre a data do fato subestimaria os anos mais recentes, porque as ocorrências que ainda serão registradas não estão lá.**Decisão.** Manter as duas datas no modelo, como dois papéis da dimensão tempo. Séries anuais usam a data da estatística; sazonalidade e horário usam a data da ocorrência. Cada consulta declara qual usa. A dimensão tempo cobre de 1970 a 2026, para que nenhum fato antigo fique órfão.---#### Não é problema — boletins com mais de uma linha**Evidência.** Cerca de 1.100 boletins aparecem em mais de uma linha, com até quatro naturezas no mesmo boletim.**Por que não é duplicidade.** É o grão do fato: um boletim que apura roubo e lesão corporal descreve dois crimes. Deduplicar por número de boletim **apagaria crimes**. A deduplicação usa a combinação de boletim, data, hora, rubrica, natureza, conduta e bairro — e remove apenas repetições idênticas.

## 3. Integridade do data warehouseAs três verificações que precisam obrigatoriamente passar para que a análise seja confiável.

In [ ]:
consultar("""
SELECT verificacao, resultado, situacao
FROM `@projeto.qualidade.verificacoes`
WHERE verificacao IN ('Carga: fato x staging',
                      'Integridade referencial: fatos órfãos',
                      'Granularidade: unicidade do grão',
                      'Domínio: município válido no IBGE',
                      'Faixa: hora da ocorrência',
                      'Faixa: coordenadas dentro de Sorocaba',
                      'Consistência: ocorrência antes do registro')
ORDER BY verificacao
""")

## 4. Conclusão da análise de qualidade**O conjunto de dados tem problemas?** Sim, sete deles — e nenhum era visível antes de perfilar atributo a atributo. Dois teriam produzido respostas erradas em silêncio (as grafias da natureza criminal e a coordenada zero tratada como número), e dois teriam inviabilizado perguntas inteiras (o período ausente e o tipo de local não publicado).**Eles afetam as respostas?** Depois do tratamento, de forma controlada e declarada:| Pergunta | Situação ||---|---|| P1, P2, P7 | **sem restrição** — apoiam-se em campos completos || P3 | sem restrição, usando a data da ocorrência || P4 | **restrita aos 75% de registros com hora**, e à granularidade de hora cheia || P5 | **restrita a 2025 e 2026**, os anos com tipo de local publicado || P6 | delegacia sem restrição; bairro com cardinalidade ainda inflada || P8 | **restrita aos 61,5% com coordenada válida** |**O que ficou sem solução.** A cardinalidade do bairro não foi inteiramente resolvida, e a ambiguidade da derivação do tipo de local é conhecida mas não eliminável. Ambos estão registrados na [autoavaliação](../docs/08-autoavaliacao.md) como trabalho futuro.**Próxima etapa:** [`04_analise_resultados.ipynb`](04_analise_resultados.ipynb) — as respostas às perguntas de negócio.